# Databricks Mosaic AI Core APIs 

## MLflow – Model & Agent Lifecycle
- **mlflow.pyfunc.log_model**: Packages an agent or model as a deployable MLflow artifact.
- **mlflow.register_model**: Registers a model or agent into Unity Catalog for governance and deployment.
- **mlflow.pyfunc.load_model**: Loads a registered model for batch inference or Python execution.
- **mlflow.set_experiment**: Sets the experiment location for runs, traces, and evaluations.
- **@mlflow.trace**: Automatically captures agent steps, tool calls, and LLM interactions.

## MLflow GenAI – Evaluation
- **mlflow.genai.evaluate**: Runs offline evaluation of agents or RAG pipelines using LLM judges.
- **mlflow.genai.datasets.create_dataset**: Creates a reusable dataset for agent evaluation.
- **mlflow.genai.datasets.get_dataset**: Loads an evaluation dataset for scoring and regression testing.
- **mlflow.genai.scorers.Correctness**: Scores factual correctness of agent responses.
- **mlflow.genai.scorers.RelevanceToQuery**: Scores how relevant a response is to the input query.
- **mlflow.genai.scorers.Guidelines**: Scores compliance with custom safety or policy rules.

## Databricks Agent Framework – Deployment
- **agents.deploy**: Deploys a Unity-Catalog-registered agent to Mosaic AI Model Serving.
- **agents.list_deployments**: Lists all active agent deployments.
- **agents.get_deployment**: Retrieves metadata and status for a deployed agent.
- **agents.delete_deployment**: Removes an agent deployment and serving endpoint.

## Unity Catalog – Governed Tools
- **DatabricksFunctionClient**: Client for registering and invoking Unity Catalog function tools.
- **create_python_function**: Registers a Python function as a governed agent tool.
- **create_function**: Registers a SQL function as a governed agent tool.

## Runtime Execution
- **mlflow.models.predict**: Executes a model or agent via its standardized predict interface.
- **mlflow.pyfunc.spark_udf**: Runs a model or agent at scale inside Spark SQL.

# Agent Evaluation
Now that we've created an agent with tools, we'll evaluate it

In [0]:
%pip install -U -qqqq mlflow-skinny[databricks] langgraph==0.3.4 databricks-langchain databricks-agents twilio python-dotenv
dbutils.library.restartPython()

In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
import sys
sys.path.append("./tools")

In [0]:
import os
from dotenv import load_dotenv, find_dotenv
import weatherwise_agent
load_dotenv(find_dotenv())  

def redact(val, show=4):
    if not val or len(val) <= show*2:
        return "****"
    return f"{val[:show]}...{val[-show:]}"

# Extract the names of all tools defined in the module
tool_names = [t.name for t in weatherwise_agent.tools]

print("Available tools:")
for name in tool_names:
    print(f" - {name}")

# read OS variables into notebook variables to be used below
TARGET_CATALOG = os.environ.get("TARGET_CATALOG")
TARGET_SCHEMA = os.environ.get("TARGET_SCHEMA")
VS_INDEX = os.environ.get("VS_INDEX")

LLM_ENDPOINT_NAME = os.environ.get("LLM_ENDPOINT_NAME")
MLFLOW_EXPERIMENT = os.environ.get("MLFLOW_EXPERIMENT")
RETRIEVER_TOOL_NAME = os.environ.get("RETRIEVER_TOOL_NAME")
AGENT_NAME = os.environ.get("AGENT_NAME")

MAILGUN_API_URL = os.environ.get("MAILGUN_API_URL")
MAILGUN_API_KEY = os.environ.get("MAILGUN_API_KEY")
SENDER = os.environ.get("SENDER")
RECIPIENT = os.environ.get("RECIPIENT")

SMS_ACCOUNT_SID = os.getenv("SMS_ACCOUNT_SID")
SMS_AUTH_TOKEN = os.getenv("SMS_AUTH_TOKEN")

print("Local variables:")
print(f" - TARGET_CATALOG: {TARGET_CATALOG}")
print(f" - TARGET_SCHEMA: {TARGET_SCHEMA}")
print(f" - LLM_ENDPOINT_NAME: {LLM_ENDPOINT_NAME}")
print(f" - MLFLOW_EXPERIMENT: {MLFLOW_EXPERIMENT}")
print(f" - VS_INDEX: {VS_INDEX}")
print(f" - RETRIEVER_TOOL_NAME: {RETRIEVER_TOOL_NAME}")
print(f" - AGENT_NAME: {AGENT_NAME}")
print(f" - MAILGUN_API_URL: {redact(MAILGUN_API_URL)}")
print(f" - MAILGUN_API_KEY: {redact(MAILGUN_API_KEY)}")
print(f" - SENDER: {SENDER}")
print(f" - RECIPIENT: {RECIPIENT}")
print(f" - SMS_ACCOUNT_SID: {redact(SMS_ACCOUNT_SID)}")
print(f" - SMS_AUTH_TOKEN: {redact(SMS_AUTH_TOKEN)}")

### 2.0 Create an MLflow Experiment

> Setting the experiment avoids MLflow using the default experiment name and keep things organized

In [0]:
import mlflow

# build a workspace path under your user home
user = spark.sql("select current_user()").first()[0]          # e.g. 'robert.leach@databricks.com'
exp_path = f"/Users/{user}/{MLFLOW_EXPERIMENT}"

# set the experiment (and end any stray run first)
mlflow.end_run()
mlflow.set_experiment(exp_path)

print("Experiment path:", exp_path)

#### 2.1 Unit Test the Agent, tracing the chain of thought of the agent

In [0]:
from weatherwise_agent import AGENT

response = AGENT.predict({
    "messages": [
        {
            "role": "user",
            "content": "I heard the weather in New York is going to be cold tomorrow. Which in-transit shipments are at risk of temperature excursions, and what supplier escalation steps should I take? Also is there a backup supplier nearby? Email me a full report and send me an super concise SMS message with an abbreviated summary"
        }
    ]
})

display(response)

### 2.2 Log the agent in MLflow (locally) before proceeding with evals

> Setting the experiment avoids MLflow using the default experiment name and keep things organized

> See [MLflow - Models from Code](https://mlflow.org/docs/latest/models.html#models-from-code).

In [0]:
import datetime
import mlflow
from weatherwise_agent import tools, LLM_ENDPOINT_NAME
from databricks_langchain import VectorSearchRetrieverTool
from mlflow.models.resources import DatabricksFunction, DatabricksServingEndpoint
from unitycatalog.ai.langchain.toolkit import UnityCatalogTool

resources = [DatabricksServingEndpoint(endpoint_name=LLM_ENDPOINT_NAME)]
for tool in tools:
    if isinstance(tool, VectorSearchRetrieverTool):
        resources.extend(tool.resources)
    elif isinstance(tool, UnityCatalogTool):
        resources.append(DatabricksFunction(function_name=tool.uc_function_name))

input_example = {
    "messages": [
        {
            "role": "user",
            "content": "I heard the weather in Philadelphia is going to be hot tomorrow. Which in-transit shipments are at risk of temperature excursions, and what supplier escalation steps should I take? Also is there a backup supplier nearby? Email me the results"
        }
    ]
}


with mlflow.start_run(run_name=f"{AGENT_NAME}-ut-{datetime.datetime.now():%Y%m%d}"):
    logged_agent_info = mlflow.pyfunc.log_model(
        name=AGENT_NAME,      # under model artifacts, the top level folder            
        python_model="weatherwise_agent.py",       # agent python file 
        input_example=input_example,
        resources=resources,
        code_paths=["tools/custom_tools/"],      
        extra_pip_requirements=[
            "databricks-connect"
        ]
    )

### 2.3 Load model into a wrapper function to make running evals easier (simplier syntax)

> Load the model and create a prediction function so you can ask the model a question without the boilerplate message: role: user: conetent: syntax

In [0]:
# Load the model and create a prediction function to ask the model a question without the boilerplate syntax of message: role: user: conetent: syntax
logged_model_uri = f"runs:/{logged_agent_info.run_id}/{AGENT_NAME}"
loaded_model = mlflow.pyfunc.load_model(logged_model_uri)

def predict_wrapper(query):
    # Format for chat-style models
    model_input = {
        "messages": [{"role": "user", "content": query}]
    }
    response = loaded_model.predict(model_input)
    
    messages = response['messages']
    return messages[-1]['content']

### 2.4 Load eval data


In [ ]:
import pandas as pd

cases = [
    # Zimmer Biotech (ZB-SUP-2041)
    {
        "request": "According to Zimmer Biotech SOP ZB-SUP-2041, outline Tier-1, Tier-2, and Tier-3 escalation contacts and the SLA.",
        "expected_facts": [
            "Zimmer Biotech SOP ZB-SUP-2041 defines tiers",
            "Tier 1: logistics@zimmerbio.com",
            "Tier 2: qa@zimmerbio.com; director.supply@zimmerbio.com",
            "Tier 3: vp.ops@zimmerbio.com | +1-312-555-0100",
            "SLA: Acknowledge 1h; CAPA 24h"
        ],
    },
    {
        "request": "Per ZB-SUP-2041, when do we escalate to Tier-2 at Zimmer Biotech?",
        "expected_facts": [
            "Tier-2 trigger: temp > 8°F over max or delay > 24h",
            "Contacts: qa@zimmerbio.com; director.supply@zimmerbio.com",
            "Tier-1 is logistics@zimmerbio.com; Tier-3 is vp.ops@zimmerbio.com",
            "SLA: Acknowledge 1h; CAPA 24h"
        ],
    },

    # MedPro Devices (MP-SLA-007)
    {
        "request": "Summarize MedPro Devices SLA MP-SLA-007 and the escalation path.",
        "expected_facts": [
            "MedPro MP-SLA-007 requires on-time ≥ 98%",
            "Zero tolerance outside 2–8°C",
            "Escalation: shipping@medpro.com → operations@medpro.com → ceo.office@medpro.com",
            "If weather delay > 12h, reroute per SLA"
        ],
    },

    # Ortho Dynamics (OD-QA-115)
    {
        "request": "For Ortho Dynamics OD-QA-115, list L1/L2/L3 escalation contacts and time targets.",
        "expected_facts": [
            "Ortho Dynamics OD-QA-115 defines L1/L2/L3",
            "L1: warehouse.alerts@orthodynamics.com (≤2h)",
            "L2: dist.ops@orthodynamics.com (≤4h)",
            "L3: cmo.office@orthodynamics.com (≤8h)"
        ],
    },
    {
        "request": "What does OD-QA-115 require when a temperature excursion is detected?",
        "expected_facts": [
            "Temp excursion → isolate lot",
            "Notify QA per OD-QA-115",
            "Follow L1/L2/L3 timing if required",
            "Document in incident log"
        ],
    },

    # CardioNova (CN-COLD-003)
    {
        "request": "List the CardioNova CN-COLD-003 escalation triggers and contact chain.",
        "expected_facts": [
            "CN-COLD-003 triggers: delay > 24h, ambient > 10°C, severe weather",
            "Chain: supply@cardionova.com → qa@cardionova.com → compliance@cardionova.com",
            "Record temperatures at handoff per storage guidance"
        ],
    },

    # ValveX Labs (VX-QMS-221)
    {
        "request": "According to ValveX Labs VX-QMS-221, classify excursions by severity and actions.",
        "expected_facts": [
            "Minor (<1°C) → log only",
            "Moderate (1–3°C) → notify QA",
            "Critical (>3°C) → isolate and escalate",
            "Contacts: qa.supervisor@valvex.com; compliance@valvex.com"
        ],
    },

    # OrthoCore (Supplier Playbook)
    {
        "request": "Summarize OrthoCore escalation rules from Supplier Playbook.",
        "expected_facts": [
            "Notify ops@orthocore.com initially",
            "If delay > 24h or temp > 8°C, escalate to qa@orthocore.com",
            "Storage: maintain 2–8°C and record at handoff"
        ],
    },

    # MedAxis (Escalation Matrix)
    {
        "request": "Provide MedAxis escalation matrix contacts (L1/L2/L3).",
        "expected_facts": [
            "MedAxis Escalation Matrix defines L1/L2/L3",
            "L1: shipping@medaxis.io",
            "L2: ops@medaxis.io",
            "L3: exec@medaxis.io"
        ],
    },

    # Cross-check product storage (common across rows)
    {
        "request": "What is the standard storage requirement mentioned for these products?",
        "expected_facts": [
            "Maintain 2–8°C (≈36–46°F)",
            "Record temperatures at handoff"
        ],
    },

    # Product-specific: Balloon Catheter, Ortho Dynamics
    {
        "request": "For the Balloon Catheter (CATH-B08) under Ortho Dynamics, what are the escalation contacts and timing?",
        "expected_facts": [
            "Product: Balloon Catheter (CATH-B08)",
            "Supplier: Ortho Dynamics (OD-QA-115)",
            "L1: warehouse.alerts@orthodynamics.com (≤2h)",
            "L2: dist.ops@orthodynamics.com (≤4h)",
            "L3: cmo.office@orthodynamics.com (≤8h)"
        ],
    },

    # Product-specific: Aortic Valve, ValveX Labs
    {
        "request": "For Aortic Valve (VALVE-E11) at ValveX Labs, how are excursions classified and who is contacted?",
        "expected_facts": [
            "Product: Aortic Valve (VALVE-E11)",
            "Supplier: ValveX Labs (VX-QMS-221)",
            "Minor <1°C log; 1–3°C notify QA; >3°C isolate & escalate",
            "qa.supervisor@valvex.com; compliance@valvex.com"
        ],
    },

    # Product-specific: Hip Implant Set at Zimmer
    {
        "request": "For Hip Implant Set (IMPL-A21) from Zimmer Biotech, list Tier-1/2/3 contacts and SLA.",
        "expected_facts": [
            "Product: Hip Implant Set (IMPL-A21) | Supplier: Zimmer Biotech",
            "Tier 1: logistics@zimmerbio.com",
            "Tier 2: qa@zimmerbio.com; director.supply@zimmerbio.com",
            "Tier 3: vp.ops@zimmerbio.com | +1-312-555-0100",
            "Acknowledge 1h; CAPA 24h"
        ],
    },

    # Product-specific: Cardio Stent Kit at MedPro
    {
        "request": "For Cardio Stent Kit (STENT-C13) at MedPro, summarize SLA and escalation ladder.",
        "expected_facts": [
            "Product: Cardio Stent Kit (STENT-C13) | Supplier: MedPro Devices",
            "On-time ≥ 98%; zero tolerance outside 2–8°C",
            "Escalation: shipping@medpro.com → operations@medpro.com → ceo.office@medpro.com",
            "Reroute if weather delay > 12h"
        ],
    },
]

eval_df = pd.DataFrame(cases)

# Format for mlflow.genai.evaluate with expected_facts as a list for Correctness scorer
eval_data = [
    {
        "inputs": {"query": row["request"]},
        "expectations": {
            "expected_facts": row["expected_facts"]
        },
    }
    for _, row in eval_df.iterrows()
]

### 2.5 Define LLM Judges


In [0]:
from mlflow.genai.scorers import (
    Correctness,         # built-in judge: checks factual accuracy vs expectations.expected_facts
    Safety,              # built-in safety scorer
    RelevanceToQuery,    # built-in relevance scorer
    Guidelines           # LLM-judge wrapper for your custom tone/style rules
)

scorers = [
    Correctness(),        # factual accuracy judge
    Safety(),             # checks harmful/off-policy content
    RelevanceToQuery(),   # measures response relevance to query
    Guidelines(           # your custom style/tone scorer
        name="jj_tone",
        guidelines="""Professional and concise:
        • No chit-chat or clarifying questions
        • No logs/JSON/system/meta
        Pass if clear, factual, and structured; fail if casual/wordy or asks questions."""
    ),
]

### 2.6 Run evals


In [0]:
print("Running evaluation...")
with mlflow.start_run(run_name=f"{AGENT_NAME}-evals-{datetime.datetime.now():%Y%m%d}"):
    results = mlflow.genai.evaluate(
        data=eval_data,
        predict_fn=predict_wrapper, 
        scorers=scorers,
    )

#### 2.6 Interate
>Based on evals go back to the agent code (.py file) and change our prompt to be more specific, more guildines, reduce marketing fluff, be more actionable, etc.

### 2.7 After satisified with evals, log model to Unity Catalog


In [0]:
from databricks.sdk import WorkspaceClient
import os

uc_model_name = f"{TARGET_CATALOG}.{TARGET_SCHEMA}.{AGENT_NAME}"

# register the model to UC
uc_registered_model_info = mlflow.register_model(model_uri=logged_agent_info.model_uri, name=uc_model_name)

In [0]:
from IPython.display import display, HTML

# Retrieve the Databricks host URL
workspace_url = spark.conf.get('spark.databricks.workspaceUrl')

# Create HTML link to created agent
html_link = f'<a href="https://{workspace_url}/explore/data/models/{TARGET_CATALOG}/{TARGET_SCHEMA}/{AGENT_NAME}" target="_blank">Go to Unity Catalog to see Registered Agent</a>'
display(HTML(html_link))

### 2.8 Deploy to model serving (and spin up human feedback app)


In [0]:
print("Local model URI:", logged_agent_info.model_uri)
print("Local model Run ID:", logged_agent_info.run_id)
print("UC Registered Model Version:", uc_registered_model_info.version)

In [0]:
from databricks import agents

# Deploy the model to the review app and a model serving endpoint


agents.deploy(
    model_name=uc_model_name,
    model_version=uc_registered_model_info.version,
    scale_to_zero=True,
    environment_vars={
        "TARGET_CATALOG": TARGET_CATALOG,
        "TARGET_SCHEMA": TARGET_SCHEMA,
        "VS_INDEX": VS_INDEX,
        "LLM_ENDPOINT_NAME": LLM_ENDPOINT_NAME,
        "RETRIEVER_TOOL_NAME": RETRIEVER_TOOL_NAME,
        "MAILGUN_API_URL": MAILGUN_API_URL,
        "MAILGUN_API_KEY": MAILGUN_API_KEY,
        "SENDER": SENDER,
        "RECIPIENT": RECIPIENT,
        "SMS_ACCOUNT_SID": SMS_ACCOUNT_SID,
        "SMS_AUTH_TOKEN": SMS_AUTH_TOKEN,
    },
    tags={"endpointSource": AGENT_NAME},
)